# Distillation pilot: does a compact student inherit a 650M RNA model's advantage, and does it survive a distribution shift?

Task: **mean ribosome load** (Sample et al. 2019, GEO GSE114002). Trained on a library of random synthetic 5'UTRs, tested on
real human 5'UTRs from the same assay, so the distribution shift is built into the data. Teacher: **RiNALMo-giga (650M)**,
official fine-tuned checkpoint (Zenodo 15043668, weights CC BY 4.0, code Apache-2.0), run through a plain-PyTorch
re-implementation because the official package hard-imports flash-attn.

The gates are pre-registered in `PILOT_PROTOCOL.md`, committed before this was run. **Stage 4 (`teacher`) must reproduce the
published R2 within 0.02 before any student result means anything.**

Runtime: **A100** (High-RAM helps). Every stage is resumable: re-run the same cell after a disconnect and it skips finished work.
Expect roughly 20-40 min for the teacher stage and 1.5-3 h for the three student grids; check the Resources tab for your
compute-unit burn rate.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 0. Files and a working directory on Drive
Upload `rnakd_pilot.zip` from the chat. Keeping `WORK` on Drive is what makes a disconnect cost minutes rather than hours.

In [ ]:
from google.colab import files, drive
up = files.upload()                      # choose rnakd_pilot.zip
!unzip -q -o rnakd_pilot.zip -d /content
%cd /content/rnakd_pilot

try:
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/rna_distill/mrl'
except Exception as e:
    print('Drive not mounted (%s); using local disk, results are lost on disconnect' % e)
    WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True); print('WORK =', WORK)

In [ ]:
!pip -q install 'numpy>=1.26' 'pandas>=2' scipy
# plumbing rehearsal on synthetic data, no network and no checkpoint: ~2 minutes, catches a broken environment early
!python -m rnakd.pilot selftest --work /content/selftest_work

## 1. Teacher checkpoint: fetch, then look at it before trusting it
`dumpkeys` prints the checkpoint's parameter names and shapes. If they do not match the re-implementation, this cell says so in
two minutes instead of a strict-load failure an hour in. 2.6 GB download.

In [ ]:
CKPT = f'{WORK}/rinalmo_giga_mrl_ft.pt'
!test -f {CKPT} || wget -q --show-progress -O {CKPT} 'https://zenodo.org/records/15043668/files/rinalmo_giga_mrl_ft.pt?download=1'
!ls -lh {CKPT}
!python -m rnakd.pilot dumpkeys --ckpt-ft {CKPT} 2>&1 | head -60

## 2. Data: the authors' splits, plus a clean in-distribution test
Downloads the GEO supplementary archive (~100 MB) and builds `train / val / iid_test / ood_test / repro_val`.
`repro_val` is the authors' Random7600, which their code uses for early stopping, so `iid_test` is the next block of
random-library UTRs selected by the same rule and never used for training or selection.

In [ ]:
!python -m rnakd.pilot data --work {WORK}

## 3. GATE P1: reproduce the published numbers, then cache the teacher once
RiNALMo-giga publishes R2 **0.925** on Random7600 and **0.860** on Human7600. This stage scores the released checkpoint through
our loader on both, and caches the predictions and features every student arm needs, so the 650M model never runs again.
It also runs a randomly initialised teacher of the same architecture to supply the `ctrl_untrained` control (drop it with
`--no-untrained` to halve this stage).

In [ ]:
# the only expensive cell: one pass of the 650M model over every split, plus a random-init pass for the control.
# Lower --bs if it runs out of memory; add --no-untrained to halve it.
!python -m rnakd.pilot teacher --work {WORK} --ckpt-ft {CKPT} --bs 128

## 4. Reference points (CPU, seconds)
`kmer_ridge` is a sequence model with no learned representation: if the students cannot beat it, nothing else in the study
matters. `probe_enc` / `probe_head` are ridge regressions on the **frozen** teacher features just cached, which separates
"the teacher's representation is informative" from "the fine-tuned teacher is accurate" - the published R2 alone cannot.

In [ ]:
!python -m rnakd.pilot baselines --work {WORK}

## 5. Students
Three grids, each resumable from `runs.csv`. `main` is the headline ladder (4 sizes x 4 arms x 3 seeds), `controls` tests whether
any gain survives a permuted or untrained teacher, `frac` asks whether distillation matters more when labels are scarce.
Students read only the cached teacher, so these cells need far less GPU than stage 3.

In [ ]:
# ~30-60 min on an A100: 4 sizes x 4 arms x 3 seeds. Safe to interrupt; re-run this cell to resume.
!python -m rnakd.pilot train --work {WORK} --grid main --bs 256

In [ ]:
!python -m rnakd.pilot train --work {WORK} --grid controls --bs 256

In [ ]:
!python -m rnakd.pilot train --work {WORK} --grid frac --bs 256

## 6. Report: the tables and the pre-registered gates
Prints the teacher reproduction, the size x arm table of IID/OOD R2 with seed spread, the retention curve, and a verdict on
P1-P5 exactly as `PILOT_PROTOCOL.md` fixed them.

In [ ]:
!python -m rnakd.pilot report --work {WORK}

In [ ]:
# only the small artifacts: the caches stay on Drive
import zipfile, os
with zipfile.ZipFile('/content/pilot_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('runs.csv', 'teacher_report.csv', 'pilot_report.txt'):
        pth = os.path.join(WORK, f)
        if os.path.exists(pth):
            z.write(pth, f)
            print('packed', f, os.path.getsize(pth), 'bytes')
files.download('/content/pilot_results.zip')   # send me these three files